# Minimal example
Run the size mitigation on a handful of spws in one call and show the final mitigation table.

Requires `smart_size_mitigation_heuristics.py` in the same directory (or on PYTHONPATH).

In [1]:
from smart_size_mitigation_heuristics import (
    build_mitigation_table,
    sizes_from_imlist,
    Limits,
    plan,
)

# 1. Define the input parameters (example case)

In [2]:
spws = [
    {  # representative spw: nbin-eligible, single pointing, robust=2.0
        "spw": "25", "nchan": 3840, "ratio_satisfied": False,
        "pb": 0.9, "is_mosaic": False, "ppb": 4.0, "robust": 2.0,
        "n_roi": 3000, "n_total": 3840,
    },
    {  # smaller nchan, only eligible for nbin if ratio condition met
        "spw": "27", "nchan": 960, "ratio_satisfied": True,
        "pb": 1.0, "is_mosaic": False, "ppb": 3.5, "robust": None,
        "n_roi": 800, "n_total": 960,
    },
    {  # nbin-eligible (nchan=7680), single pointing
        "spw": "29", "nchan": 7680, "ratio_satisfied": False,
        "pb": 0.8, "is_mosaic": False, "ppb": 4.0, "robust": None,
        "n_roi": 1000, "n_total": 7680,
    },
    {  # not nbin-eligible at all (ratio condition not met), tight PB already
        "spw": "31", "nchan": 1920, "ratio_satisfied": False,
        "pb": 0.65, "is_mosaic": False, "ppb": 5.0, "robust": 2.0,
        "n_roi": 1200, "n_total": 1920,
    },
]

# The sizes are not typed in: they are computed from the imaging target list
# (in the pipeline: the targets of makeimlist's result) with calculate_sizes().
# Here: 3 fields, each imaged in every spw.
imsizes = {"25": 990, "27": 1980, "29": 1510, "31": 1400}
imlist = [
    {"field": field, "spw": spw, "imsize": (n, n), "nbin": -1, "specmode": "cube"}
    for field in ("f1", "f2", "f3")
    for spw, n in imsizes.items()
]
get_nchan = {row["spw"]: row["nchan"] for row in spws}.__getitem__  # spw -> nchan

# 2. Compute the sizes and build the per-lever mitigation table

In [3]:
cube_sizes, product_sizes, nfields = sizes_from_imlist(imlist, get_nchan)
print("cube sizes (GB):   ", {k: round(v, 1) for k, v in cube_sizes.items()})
print("product sizes (GB):", {k: round(v, 1) for k, v in product_sizes.items()})
print("nfields:", nfields)

table = build_mitigation_table(spws, use_roi=False, nfields=nfields)
table

cube sizes (GB):    {'25': 15.1, '27': 15.1, '29': 70.0, '31': 15.1}
product sizes (GB): {'25': 90.3, '27': 90.4, '29': 420.3, '31': 90.4}
nfields: 3


,n_bin,PB,cell,ROI,nfields_total,nfields_kept,nfields,total_size_fraction
spw,,,,,,,,
25,0.5,0.604938,0.660156,1.0,3,3,0.333333,0.066559
27,0.5,0.490000,0.734694,1.0,3,3,0.333333,0.060000
29,0.5,0.765625,0.562500,1.0,3,3,0.333333,0.071777
31,1.0,1.000000,0.422500,1.0,3,3,0.333333,0.140833


# 3. Set the size limits (GB)

In [9]:
limits = Limits(maxcubesize=40.0, maxproductsize=500.0, maxcubelimit=100.0)
limits


Limits(maxcubesize=40.0, maxproductsize=500.0, maxcubelimit=100.0)

# 4. Run the mitigation in one call. spw "29" is the representative spw here
Every possible plan is scored and the one with the highest score is returned. `return_candidates=True` additionally returns the full list of plans (step 6).

In [10]:
final, msg = plan(
    table,
    limits,
    imlist,
    get_nchan,
    repr_spw="29",
    return_candidates=True,  # for the diagnostic list below
)

# 5. Display the outcome.

In [11]:
print(f"Result: {msg['shortmsg']}")
print(f"\nFinal score: {msg['score']:.3f}  (1 = no mitigation needed, 0 = limits cannot be met)")
print(msg["scores"].round(3).to_string())
print(f"\nSpws kept: {list(final.index[final['kept']])}")

Result: Size was mitigated

Final score: 0.998  (1 = no mitigation needed, 0 = limits cannot be met)
n_bin     0.998
PB        1.000
cell      1.000
ROI       1.000
spws      1.000
fields    1.000

Spws kept: ['25', '27', '29', '31']


In [12]:
final

,n_bin,PB,cell,ROI,nfields_total,nfields_kept,nfields,total_size_fraction,cube_size_GB,product_size_GB,required_size_fraction,min_steps,levers_used,achieved_size_fraction,field_size_fraction,mitigated_cube_GB,mitigated_product_GB,kept
spw,,,,,,,,,,,,,,,,,,
25,0.5,0.604938,0.660156,1.0,3,3,0.333333,0.066559,15.054336,90.349538,1.000000,0,-,1.0,1.0,15.054336,90.349538,True
27,0.5,0.490000,0.734694,1.0,3,3,0.333333,0.060000,15.054336,90.420106,1.000000,0,-,1.0,1.0,15.054336,90.420106,True
29,0.5,0.765625,0.562500,1.0,3,3,0.333333,0.071777,70.044672,420.322754,0.571064,1,n_bin,0.5,1.0,35.022336,210.161377,True
31,1.0,1.000000,0.422500,1.0,3,3,0.333333,0.140833,15.052800,90.363840,1.000000,0,-,1.0,1.0,15.052800,90.363840,True


# 6. Diagnostics: all plans, sorted by score
One row per plan: the cube levers applied to each spw (`dropped` = spw removed), fields kept, resulting sizes (GB), whether all limits are met, the lever scores and the final `score` (0 if infeasible).

In [13]:
candidates = msg["candidates"]
print(f"{len(candidates)} plans, {candidates['feasible'].sum()} feasible")
candidates.head(50).round(3)

4374 plans, 4373 feasible


,29,25,27,31,n_spws_kept,nfields_kept,n_levers,product_GB,max_cube_GB,feasible,n_bin,PB,cell,ROI,spws,fields,score
0,n_bin,-,-,-,4,3,1,481.295,35.022,True,0.998,1.000,1.000,1.0,1.0,1.0,0.998
1,"PB,cell",-,-,-,4,3,2,452.151,30.166,True,1.000,0.999,0.999,1.0,1.0,1.0,0.998
2,cell,-,cell,-,4,3,2,483.576,39.400,True,1.000,1.000,0.998,1.0,1.0,1.0,0.998
3,"n_bin,PB",-,-,-,4,3,2,432.038,26.814,True,0.998,0.999,1.000,1.0,1.0,1.0,0.998
4,cell,cell,-,-,4,3,2,476.860,39.400,True,1.000,1.000,0.998,1.0,1.0,1.0,0.998
5,n_bin,-,cell,-,4,3,2,457.306,35.022,True,0.998,1.000,0.999,1.0,1.0,1.0,0.998
6,cell,PB,-,-,4,3,2,471.871,39.400,True,1.000,0.999,0.999,1.0,1.0,1.0,0.997
7,n_bin,cell,-,-,4,3,2,450.590,35.022,True,0.998,1.000,0.999,1.0,1.0,1.0,0.997
8,"PB,cell",-,cell,-,4,3,3,428.162,30.166,True,1.000,0.999,0.998,1.0,1.0,1.0,0.997
9,n_bin,PB,-,-,4,3,2,445.601,35.022,True,0.998,0.999,1.000,1.0,1.0,1.0,0.997
